In [2]:
# ProoV: VW & Audi Used Car Price Prediction
# Volkswagen, Audi, VW, Golf, A4 are trademarks of Volkswagen AG.
# Independent educational case study. Not affiliated with VW AG.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})
sns.set_palette('muted')

print('Libraries loaded successfully.')

import IPython.display
def proov_tick(task_id):
    IPython.display.display(IPython.display.Javascript(f'window.parent.postMessage({{ type: "PROOV_CHECKPOINT", checkpoint: "{task_id}" }}, "*");'))
proov_tick('setup_complete')


Libraries loaded successfully.


<IPython.core.display.Javascript object>

In [3]:
import pandas as pd

print("Loading datasets from local files...")

vw = pd.read_csv("vw.csv")
audi = pd.read_csv("audi.csv")

print(f"✓ Loaded vw.csv: {len(vw):,} rows")
print(f"✓ Loaded audi.csv: {len(audi):,} rows")
print(f"\nTotal: {len(vw) + len(audi):,} listings ready for analysis.")

Loading datasets from local files...
✓ Loaded vw.csv: 15,157 rows
✓ Loaded audi.csv: 10,668 rows

Total: 25,825 listings ready for analysis.


In [4]:
# Quick sanity check
print('=== VW Dataset ===')
print(f'Shape: {vw.shape}')
display(vw.head(3))

print('\n=== Audi Dataset ===')
print(f'Shape: {audi.shape}')
display(audi.head(3))

=== VW Dataset ===
Shape: (15157, 9)


,model,year,price,transmission,mileage,fuelType,tax,mpg,engineSize
0,T-Roc,2019,25000,Automatic,13904,Diesel,145,49.6,2.0
1,T-Roc,2019,26883,Automatic,4562,Diesel,145,49.6,2.0
2,T-Roc,2019,20000,Manual,7414,Diesel,145,50.4,2.0



=== Audi Dataset ===
Shape: (10668, 9)


,model,year,price,transmission,mileage,fuelType,tax,mpg,engineSize
0,A1,2017,12500,Manual,15735,Petrol,150,55.4,1.4
1,A6,2016,16500,Automatic,36203,Diesel,20,64.2,2.0
2,A1,2016,11000,Manual,29946,Petrol,30,55.4,1.4


In [5]:
df = pd.concat([vw.assign(brand='VW'), audi.assign(brand='Audi')], ignore_index=True)
print('Sprint 1 Prep: df is ready with VW and Audi combined.')

Sprint 1 Prep: df is ready with VW and Audi combined.


In [6]:
before = len(df)

df = df[(df['engineSize'] > 0) &
        (df['price'] >= 500) &
        (df['price'] <= 100000)].copy()

after = len(df)

print(f'Removed {before - after:,} rows  ({before:,} -> {after:,})')

assert df['engineSize'].min() > 0
assert df['price'].min() >= 500
assert df['price'].max() <= 100000

print('Validation passed.')

proov_tick('task_2_1')

Removed 90 rows  (25,825 -> 25,735)
Validation passed.


<IPython.core.display.Javascript object>

In [7]:

df['car_age'] = 2020 - df['year']

df['mileage_per_year'] = df['mileage'] / df['car_age']

df['mileage_per_year'] = df['mileage_per_year'].replace([np.inf, -np.inf], 0).fillna(0)

assert 'car_age' in df.columns
assert 'mileage_per_year' in df.columns
assert df['mileage_per_year'].isna().sum() == 0

print(f'car_age range: {df["car_age"].min():.0f}-{df["car_age"].max():.0f} years')
print('Validation passed.')

proov_tick('task_2_2')

car_age range: 0-23 years
Validation passed.


<IPython.core.display.Javascript object>

In [8]:
y = df['price']

X = df.drop(columns=['model', 'year', 'price'])

X = pd.get_dummies(
    X,
    columns=['transmission', 'fuelType', 'brand'],
    drop_first=True
)

assert 'X' in dir()
assert 'price' not in X.columns
assert X.isna().sum().sum() == 0

print(f'Feature matrix X: {X.shape[0]:,} rows x {X.shape[1]} features')
print(f'Target y: min=£{y.min():,.0f}  max=£{y.max():,.0f}  mean=£{y.mean():,.0f}')

proov_tick('task_2_3')

Feature matrix X: 25,735 rows x 12 features
Target y: min=£899  max=£97,500  mean=£19,246


<IPython.core.display.Javascript object>